In [ ]:
# 학습된 대형모델(백본)인 MobileNet v2를 전이학습하여 cifar10 dataset을 분류하기
# 전이학습(Transfer Learning) : 이미 학습된 모델을 일부 재학습하여 새로운 작업에 적용하는 기법입니다.

import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 정규화
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 인코딩
NUM_CLASSES = 10
y_train = tf.keras.utils.to_categorical(y_train, NUM_CLASSES) # 원핫인코딩
y_test = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

print('train data : ', x_train.shape, y_train.shape) # (50000, 32, 32, 3) (50000, 10)
print('test data : ', x_test.shape, y_test.shape) # (10000, 32, 32, 3) (10000, 10)

In [ ]:
# 전익학습 : 기존 모델(백본)의 가중치는 모두 동결(freeze)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),  # 입력 크기 96, 128, 160, 192, 224...을 지원
    include_top=False,   # MobileNetV2의 분류기 부분 제거 -> 컨볼루션 레이어만 남음
    weights='imagenet',  # 사전 학습된 가중치 호출(120만 이미지, 1000개의 클래스)
)

base_model.trainable = False  # MobileNetV2는 학습에 참여하지 않도록 동결

print(base_model.summary())
#  Total params: 2,257,984 (8.61 MB)
#  Trainable params: 0 (0.00 B)
#  Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# MobileNetV2를 이용해 새로운 모델 생성
inputs = tf.keras.Input(shape=(32, 32, 3))
x = tf.keras.layers.Resizing(128, 128)(inputs)  # cifar10 이미지를 MobileNetV2에 맞게 리사이징
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x) # MaxPooling2D보다 더 급격하게 feature의 크기를 줄임
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model_tl = tf.keras.Model(inputs, outputs)
model_tl.summary()

model_tl.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model_tl.fit(x_train, y_train, batch_size=64, epochs=10, validation_split=0.2, verbose=2)

loss, accuracy = model_tl.evaluate(x_test, y_test, verbose=0)
print('Test loss:', loss)
print('Test accuracy:', accuracy)

In [ ]:
import matplotlib.pyplot as plt

# 성능 시각화
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], 'b-', label='loss')
plt.plot(history.history['val_loss'], 'r--', label='val_loss')
plt.xlabel('Epoch')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], 'g-', label='accuracy')
plt.plot(history.history['val_accuracy'], 'k--', label='val_accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
# 미세조정 (Fine Tunning) : 전이 학습 이후 성능 향상을 위해 백본의 일부 층을 학습에 참여시킴
base_model.trainable = True
for layer in base_model.layers[:-10]:  # 마지막 10개 층만 해제하고 나머지는 다시 동결
  layer.trainable = False

model_tl.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])  # 학습률이 매우 적어야함.
history = model_tl.fit(x_train, y_train, batch_size=64, epochs=10, validation_split=0.2, verbose=2)

loss, accuracy = model_tl.evaluate(x_test, y_test, verbose=0)
print('Test loss:', loss)
print('Test accuracy:', accuracy)